# LLM APIs, Reasoning Flows & Prompt-to-Action Pipelines

**Track:** Applied Agent Engineering Foundations  
**Module:** LLM Foundations  
**Environment:** SageMaker Jupyter Notebook  
**Audience:** Software engineers building enterprise AI workflows on AWS

## What learners will learn
By the end of this lab, learners will be able to:
1. Connect securely from SageMaker to Amazon Bedrock.
2. Discover available Bedrock foundation models from code.
3. Use Bedrock LLM APIs for prompt experiments and structured generation.
4. Build a prompt-to-action pipeline with tool/function-calling style behavior.
5. Validate model output before execution.
6. Manage context windows using chunking and budget checks.
7. Add safe error handling and run logging for enterprise usage.

## Instructor flow
- Part A: Secure setup and model discovery
- Part B: Prompt styles and reasoning flows
- Part C: Prompt-to-action pipeline and tool/function calling pattern
- Part D: Context window management
- Part E: Error handling, validation, and mini-hack

## Before you run

This notebook assumes:
- you are running inside **SageMaker Jupyter Notebook**
- your notebook has an **execution role** with Bedrock and S3 read permissions
- Bedrock model access is already enabled for the models you want to test
- your learning files are already uploaded to **S3**
- you are **reading** from S3 only in this notebook; no S3 write-back is required

### Design choice for this lab
This notebook teaches a **model-agnostic prompt-to-action pipeline**.  
That means learners first understand:
- how to call an LLM API
- how to get structured output
- how to validate it
- how to safely call backend functions

Later, the same pattern can be extended to native tool use, agents, workflows, or orchestration frameworks.

In [1]:
# Uncomment only if your environment is missing any package
# %pip install -q boto3 pandas

import json
import re
import textwrap
from typing import Any, Dict, List, Optional

import boto3
import pandas as pd
from botocore.exceptions import ClientError


# Get AWS_REGION from the current boto3 session
AWS_REGION = "us-west-2"

# Bedrock clients
bedrock = boto3.client("bedrock", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

# Other AWS clients we will use in this notebook
s3 = boto3.client("s3", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)


# Print out the configuration to verify everything is set up correctly
print("AWS Region:", AWS_REGION)

AWS Region: us-west-2


## Step 1 — Show available models in Bedrock

**Goal:** Let learners see what Bedrock exposes in the current region.

**Discuss:**
- model access can vary by account and region
- some models support chat, some embeddings, some image or multimodal inputs
- classroom code should **discover** available models instead of relying on memory

In [2]:
def list_bedrock_models() -> pd.DataFrame:
    response = bedrock.list_foundation_models()
    rows = []

    for item in response.get("modelSummaries", []):
        rows.append({
            "provider": item.get("providerName"),
            "model_id": item.get("modelId"),
            "model_name": item.get("modelName"),
            "input_modalities": ", ".join(item.get("inputModalities", [])),
            "output_modalities": ", ".join(item.get("outputModalities", [])),
            "streaming": item.get("responseStreamingSupported"),
            "customizations": ", ".join(item.get("customizationsSupported", [])),
            "inference_types": ", ".join(item.get("inferenceTypesSupported", [])),
        })

    df = pd.DataFrame(rows).sort_values(["provider", "model_id"]).reset_index(drop=True)
    return df

try:
    models_df = list_bedrock_models()
    with pd.option_context('display.max_rows', 250):
        display(models_df)
except ClientError as e:
    print("Could not list models. Check Bedrock permissions and regional access.")
    print(e)

,provider,model_id,model_name,input_modalities,output_modalities,streaming,customizations,inference_types
0,Amazon,amazon.nova-2-lite-v1:0,Nova 2 Lite,"TEXT, IMAGE, VIDEO",TEXT,True,,INFERENCE_PROFILE
1,Amazon,amazon.nova-2-sonic-v1:0,Nova 2 Sonic,SPEECH,"SPEECH, TEXT",True,,ON_DEMAND
2,Amazon,amazon.nova-lite-v1:0,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True,,INFERENCE_PROFILE
3,Amazon,amazon.nova-micro-v1:0,Nova Micro,TEXT,TEXT,True,,INFERENCE_PROFILE
4,Amazon,amazon.nova-premier-v1:0,Nova Premier,"TEXT, IMAGE, VIDEO",TEXT,True,,INFERENCE_PROFILE
5,Amazon,amazon.nova-premier-v1:0:1000k,Nova Premier,"TEXT, IMAGE, VIDEO",TEXT,True,,
6,Amazon,amazon.nova-premier-v1:0:20k,Nova Premier,"TEXT, IMAGE, VIDEO",TEXT,True,,
7,Amazon,amazon.nova-premier-v1:0:8k,Nova Premier,"TEXT, IMAGE, VIDEO",TEXT,True,,
8,Amazon,amazon.nova-premier-v1:0:mm,Nova Premier,"TEXT, IMAGE, VIDEO",TEXT,True,,
9,Amazon,amazon.nova-pro-v1:0,Nova Pro,"TEXT, IMAGE, VIDEO",TEXT,True,,INFERENCE_PROFILE


## Step 2 — Secure endpoint usage and minimal smoke test

**Goal:** Confirm that SageMaker can call Bedrock safely.

**Secure usage principles:**
- do **not** hardcode keys in notebooks
- use the **SageMaker execution role**
- restrict model IDs through config or an allowlist
- start with a **small, cheap** smoke test
- fail early if the model ID is not approved

**Why this matters:**  
In enterprise settings, the first failure should happen **before** a costly or unsafe request is sent.

In [3]:
# Choose one classroom model that supports text generation through Converse.
# Available model for this class
"""
amazon.nova-lite-v1:0 
amazon.nova-micro-v1:0 
"""
BEDROCK_TEXT_MODEL = "arn:aws:bedrock:us-west-2:502761807248:application-inference-profile/7sgbtd923sy4"

print("Default text model:", BEDROCK_TEXT_MODEL)
print("Caller ARN:", sts.get_caller_identity()["Arn"])

Default text model: arn:aws:bedrock:us-west-2:502761807248:application-inference-profile/7sgbtd923sy4
Caller ARN: arn:aws:sts::502761807248:assumed-role/SageMakerExecutionRole-S3Controlled/SageMaker


In [4]:
# Call LLM and get a response
response = bedrock_runtime.converse(
    modelId=BEDROCK_TEXT_MODEL,
    system=[{"text": "You are a helpful enterprise AI assistant."}],
    messages=[
        {
        "role": "user",
        "content": [{"text": "In two lines, summarize the plot of the movie Inception."}]}
    ]   ,
    inferenceConfig={
    "maxTokens": 250,
    "temperature": 0.5,
    "topP": 0.9
    }
)

print(response)

{'ResponseMetadata': {'RequestId': 'efbbccaf-6d20-4762-95fa-12156c8e9df0', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 19 May 2026 12:03:34 GMT', 'content-type': 'application/json', 'content-length': '418', 'connection': 'keep-alive', 'x-amzn-requestid': 'efbbccaf-6d20-4762-95fa-12156c8e9df0'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'text': '"Inception" follows Dom Cobb, a skilled thief who steals secrets from within the subconscious during the dream world, as he is tasked with planting an idea in a target\'s mind through a complex, multi-layered heist.'}]}}, 'stopReason': 'end_turn', 'usage': {'inputTokens': 20, 'outputTokens': 47, 'totalTokens': 67}, 'metrics': {'latencyMs': 354}}


In [5]:
print(response["usage"])

{'inputTokens': 20, 'outputTokens': 47, 'totalTokens': 67}


In [6]:
print(response["ResponseMetadata"])

{'RequestId': 'efbbccaf-6d20-4762-95fa-12156c8e9df0', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 19 May 2026 12:03:34 GMT', 'content-type': 'application/json', 'content-length': '418', 'connection': 'keep-alive', 'x-amzn-requestid': 'efbbccaf-6d20-4762-95fa-12156c8e9df0'}, 'RetryAttempts': 0}


In [7]:
print(response["output"])

{'message': {'role': 'assistant', 'content': [{'text': '"Inception" follows Dom Cobb, a skilled thief who steals secrets from within the subconscious during the dream world, as he is tasked with planting an idea in a target\'s mind through a complex, multi-layered heist.'}]}}


In [8]:
print(response["output"]["message"]["content"][0]["text"])

"Inception" follows Dom Cobb, a skilled thief who steals secrets from within the subconscious during the dream world, as he is tasked with planting an idea in a target's mind through a complex, multi-layered heist.


In [9]:
# Now do the same with a helper function that we can reuse for the rest of the notebook
def ask_llm(user_prompt: str,
            system_prompt: str = "You are a helpful enterprise AI assistant.",
            model_id: str = BEDROCK_TEXT_MODEL,
            max_tokens: int = 250,
            temperature: float = 0.2,
            top_p: float = 0.9) -> str:

    response = bedrock_runtime.converse(
        modelId=model_id,
        system=[{"text": system_prompt}],
        messages=[
            {
                "role": "user",
                "content": [{"text": user_prompt}]
            }
        ],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": temperature,
            "topP": top_p
        }
    )
    return response["output"]["message"]["content"][0]["text"]


In [10]:
# Test the helper function with a new prompt
print(ask_llm("In two lines, explain what an LLM API call is."))

An LLM API call is a request to a machine learning model's interface to generate, analyze, or transform text based on user input, utilizing large language models for tasks like text completion, summarization, or translation.


In [11]:
# Change the parameters to see how the response changes
print(ask_llm("In two lines, explain what an LLM API call is.",max_tokens=100, temperature=0.5))

An LLM API call is a request to a language model service, typically used to generate, analyze, or transform text based on the model's trained capabilities. It involves sending a query or input to the API and receiving a response or output from the language model.


In [12]:
# Activity: Change prompt parameters
# Task:
# Try the same prompt with:
# 1. temperature = 0
# 2. temperature = 0.7
# 3. max_tokens = 50
# 4. top_p = 0.5
# Compare the outputs.

prompt = "Explain why LLM validation matters in enterprise applications."

In [13]:
# Activity: 
# Ask the model about a very recent or real-time event.
# Example:
# - "What are the latest news updates today?"
# - "Who won yesterday's IPL match?"
# - "What is happening in global markets right now?"


## Step 3 — Prompt styles and reasoning flows

We will test three common patterns:
1. simple instruction
2. role-based instruction
3. structured reasoning flow

**Explain to learners:**  
Better prompts are not just longer prompts.  
Better prompts make the model's job easier by clarifying:
- role
- task
- output shape
- constraints
- reasoning path

In [14]:
# Simple instruction prompt style
prompt = "Summarize why validation matters in enterprise AI systems."
print(ask_llm(prompt))

Validation is crucial in enterprise AI systems for several reasons:

1. **Accuracy and Reliability**: Validation ensures that the AI models produce accurate and reliable outputs, which is essential for making informed business decisions and maintaining customer trust.

2. **Performance Metrics**: It helps in assessing key performance indicators (KPIs) such as precision, recall, and F1 score, ensuring that the model meets the required standards for its intended use.

3. **Error Detection**: Validation processes help identify and rectify errors, biases, and anomalies in the data and model, which can lead to more robust and fair AI systems.

4. **Compliance and Governance**: In regulated industries, validation ensures that AI systems comply with legal and ethical standards, reducing the risk of non-compliance penalties and reputational damage.

5. **Resource Optimization**: By validating models early in the development process, enterprises can avoid costly mistakes and resource wastage th

In [15]:
# Role based prompt style
prompt = """You are an enterprise AI expert. 
            Summarize why validation matters in enterprise AI systems."""
print(ask_llm(prompt))

Validation is a critical component in enterprise AI systems for several reasons:

1. **Accuracy and Reliability**: Validation ensures that the AI models produce accurate and reliable results. By rigorously testing the models against known datasets and real-world scenarios, enterprises can be confident that the AI will perform as expected in production.

2. **Compliance and Risk Management**: Many enterprise AI systems operate within regulatory frameworks. Validation helps ensure that the AI complies with legal and industry standards, thereby mitigating legal and reputational risks.

3. **Performance Optimization**: Validation processes often involve tuning and optimizing the AI models to improve their performance metrics such as precision, recall, and F1 score. This optimization is crucial for achieving the desired outcomes and maintaining competitive advantage.

4. **Trust and Adoption**: Stakeholders, including executives, clients, and end-users, are more likely to trust and adopt AI

In [16]:
# Reasoning flow prompt style
prompt = """You are an enterprise AI expert. 
            Summarize why validation matters in enterprise AI systems.
            Explain your reasoning step by step."""
print(ask_llm(prompt))

Certainly! Validation is a critical component in enterprise AI systems for several reasons. Let's break down the reasoning step by step:

### 1. **Ensuring Accuracy and Reliability**
   - **Reasoning**: AI models make predictions or decisions based on data. If these predictions are inaccurate, the consequences can be severe, especially in critical domains like finance, healthcare, or safety.
   - **Explanation**: Validation helps to assess how well the model performs on unseen data, ensuring that it generalizes well and maintains accuracy across different scenarios.

### 2. **Preventing Overfitting**
   - **Reasoning**: Overfitting occurs when a model learns the training data too well, including noise and outliers, which leads to poor performance on new data.
   - **Explanation**: Validation sets or techniques like cross-validation help in detecting overfitting by providing a separate dataset to evaluate the model’s performance, ensuring it doesn’t just memorize the training data.

###

In [17]:

# Activity: Compare prompt styles
# Task:
# Write one simple prompt and one role-based prompt for the same question.
# Compare which answer is more useful.
# You can also play with temperature and max tokens

# Your code goes here


## System Prompts

In [18]:
# System message prompt style
prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = "You are an enterprise AI expert. Please provide a detailed answer."        
print(ask_llm(prompt, system_prompt=system_prompt))

Validation is a critical component in the development and deployment of enterprise AI systems for several reasons:

### 1. **Accuracy and Reliability**
Validation ensures that the AI models are accurate and reliable in their predictions or decisions. This is crucial for enterprise AI systems that are often used in high-stakes environments such as finance, healthcare, and logistics. Accurate models lead to better decision-making and reduce the risk of costly errors.

### 2. **Compliance and Regulatory Adherence**
Many industries are subject to strict regulatory requirements. Validation helps ensure that AI systems comply with these regulations. For example, in healthcare, AI systems must adhere to HIPAA regulations, and in finance, they must comply with GDPR and other financial regulations. Validation processes often include checks to ensure that the AI system's operations are transparent and auditable, which is necessary for compliance.

### 3. **Risk Management**
Validation helps in i

In [19]:
# Another example of System message prompt style with a more specific system prompt
prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are a seasoned enterprise AI consultant with 20 years of experience.
                     You have worked with Fortune 500 companies across various industries to help them implement and validate their AI systems effectively. 
                     Please provide a detailed answer on why validation is crucial in enterprise AI systems, drawing from your extensive experience."""     
print(ask_llm(prompt, system_prompt=system_prompt))

Validation is a cornerstone in the implementation and deployment of enterprise AI systems for several critical reasons, which I'll outline based on my extensive experience working with Fortune 500 companies across diverse industries:

### 1. **Ensuring Accuracy and Reliability**
Validation ensures that AI models produce accurate and reliable results. In enterprise settings, decisions often hinge on the insights derived from AI systems. For example, in finance, an AI model used for fraud detection must accurately identify fraudulent transactions without false positives. Inaccurate predictions can lead to significant financial losses or operational inefficiencies.

### 2. **Compliance and Regulatory Adherence**
Many industries are heavily regulated, such as healthcare, finance, and telecommunications. Validation helps ensure that AI systems comply with industry regulations and standards. For instance, in healthcare, AI systems used for diagnostic purposes must adhere to stringent regulat

In [20]:
# More complex prompt styles with few-shot examples can also be used
prompt = "You are an enterprise AI expert."
system_prompt = """You are an enterprise AI expert.
Here are some examples of how to answer questions about enterprise AI systems:                  
Q: Why is validation important in enterprise AI systems?
A: Validation is crucial in enterprise AI systems to ensure that the models are performing as expected,
    to identify any biases or issues before deployment, and to maintain trust with stakeholders by demonstrating that the AI system is reliable and effective.  
Q: What are some common techniques for validating enterprise AI systems?    
A: Common techniques for validating enterprise AI systems include cross-validation, A/B testing, monitoring performance metrics in production, and conducting regular audits to check for bias and drift."""
print(ask_llm(prompt, system_prompt=system_prompt))


Certainly! Here are some more examples of how to answer questions about enterprise AI systems:

---

**Q: What are the key considerations for implementing enterprise AI systems?**

**A:** When implementing enterprise AI systems, several key considerations must be taken into account:

1. **Data Quality and Availability:** Ensuring high-quality, clean, and relevant data is crucial for the success of AI models. This includes data governance, data integration, and data labeling.

2. **Model Selection and Performance:** Choosing the right algorithms and models that align with business objectives and performance metrics is essential. This involves evaluating various models through experimentation and validation.

3. **Scalability and Infrastructure:** The AI system must be scalable to handle increasing amounts of data and users without performance degradation. This includes selecting appropriate cloud services or on-premises infrastructure.

4. **Compliance and Ethics:** Ensuring that the AI

In [21]:
# Change system prompt to agent tell the model it is an agent that needs to answer the question and take action if needed.
prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are an enterprise AI agent. 
                Your task is to answer the user's question."""
print(ask_llm(prompt, system_prompt=system_prompt))

Validation is crucial in enterprise AI systems for several reasons:

1. **Accuracy and Reliability**: Validation ensures that the AI models produce accurate and reliable outputs. This is essential for making informed business decisions and maintaining customer trust.

2. **Performance Measurement**: It helps in measuring the performance of AI models against predefined metrics, ensuring they meet the required standards and expectations.

3. **Error Detection**: Validation identifies errors, biases, and anomalies in the AI system, allowing for timely corrections and improvements.

4. **Compliance and Risk Management**: It ensures that the AI systems comply with regulatory requirements and mitigates risks associated with incorrect or biased predictions.

5. **User Trust**: Consistent validation builds user confidence in the AI system’s capabilities, leading to better adoption and usage.

6. **Iterative Improvement**: It facilitates continuous improvement by providing feedback loops that i

In [22]:
#  Activity: Rewrite system prompt
# Task:
# Change the system prompt so the model answers as:
# 1. QA lead
# 2. Product manager
# 3. Security reviewer

user_prompt = "What risks should we check before deploying an LLM application?"

# Your code goes here 

# LLM Interaction Logging & Observability Framework

Build a reusable LLM interface that not only generates responses but also captures structured telemetry for monitoring, cost tracking, and analysis.

In [23]:
import os
import pandas as pd
from datetime import datetime

LOG_CSV_PATH = "llm_prompt_log.csv"

def ask_llm(
    user_prompt: str,
    asked_by: str,
    system_prompt: str = "You are a helpful enterprise AI assistant.",
    model_id: str = BEDROCK_TEXT_MODEL,
    max_tokens: int = 250,
    temperature: float = 0.2,
    csv_path: str = LOG_CSV_PATH
) -> str:
    """
    Call the Bedrock model, return the answer,
    and log prompt/response details into a CSV using pandas.
    """

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    response = bedrock_runtime.converse(
        modelId=model_id,
        system=[{"text": system_prompt}],
        messages=[
            {
                "role": "user",
                "content": [{"text": user_prompt}]
            }
        ],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": temperature
        }
    )

    # Extract answer text
    answer = response["output"]["message"]["content"][0]["text"]

    # Extract usage safely
    usage = response.get("usage", {})
    input_tokens = usage.get("inputTokens", None)
    output_tokens = usage.get("outputTokens", None)
    total_tokens = usage.get("totalTokens", None)

    # Create one log row
    new_row = pd.DataFrame([{
        "timestamp": timestamp,
        "asked_by": asked_by,
        "model_id": model_id,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "response_text": answer,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "max_tokens": max_tokens,
        "temperature": temperature
    }])

    # Append to existing CSV or create new one
    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path)
        updated_df = pd.concat([existing_df, new_row], ignore_index=True)
    else:
        updated_df = new_row

    updated_df.to_csv(csv_path, index=False)

    return answer

In [24]:
user_prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are an enterprise AI agent. 
                Your task is to answer the user's question."""
user = "user@123"
ask_llm(user_prompt, system_prompt=system_prompt,asked_by=user)

'Validation is crucial in enterprise AI systems for several reasons:\n\n1. **Accuracy and Reliability**: Validation ensures that the AI models produce accurate and reliable results, which is essential for making informed business decisions.\n\n2. **Performance Measurement**: It helps in measuring the performance of the AI models against predefined metrics, ensuring they meet the required standards.\n\n3. **Risk Mitigation**: By validating AI systems, enterprises can identify and mitigate potential risks associated with incorrect predictions or decisions, thus safeguarding against financial and reputational damage.\n\n4. **Compliance and Standards**: Validation ensures that AI systems comply with industry regulations and standards, which is often a requirement for legal and operational reasons.\n\n5. **User Trust**: Consistent and validated AI outputs build user trust, which is critical for customer satisfaction and retention.\n\n6. **Iterative Improvement**: Validation provides feedbac

In [25]:
# Activity:
# Please try asking the model a few more questions with different system prompts and parameters to generate more rows in the log for analysis.
# Inspect the LLM log
# Task:
# Display the CSV log and compare token usage across prompts.

# user_prompt = ""
# system_prompt = ""
# user = ""
# ask_llm(user_prompt, system_prompt=system_prompt,asked_by=user)

log_df = pd.read_csv(LOG_CSV_PATH)
display(log_df)

,timestamp,asked_by,model_id,system_prompt,user_prompt,response_text,input_tokens,output_tokens,total_tokens,max_tokens,temperature
0,2026-05-19 12:04:07,user@123,arn:aws:bedrock:us-west-2:502761807248:applica...,You are an enterprise AI agent. \n ...,Summarize why validation matters in enterprise...,Validation is crucial in enterprise AI systems...,31,205,236,250,0.2


## Reflection checkpoint
Discuss pros and cons of prompt style before move to next section

## Step 4 — Build a prompt-to-action pipeline

A production-safe workflow often looks like this:
1. understand the request
2. convert it into a structured plan
3. validate the plan
4. execute only the allowed action
5. call a controlled backend function
6. log the result

This is the core **tool/function calling pattern** we want learners to understand.

### Important note
This notebook uses a **model-agnostic structured JSON plan**.  
That keeps the concept simple and portable across models.

In [26]:
ACTION_PLANNER_PROMPT = '''
You are an action planner for an enterprise assistant.

Return valid JSON only.
Use this schema:
{
  "intent": "<one short label>",
  "needs_backend_action": true or false,
  "action_name": "<action or none>",
  "arguments": { ... },
  "reason_for_action": "<one sentence>"
}

Allowed action names:
- create_ticket
- lookup_user_record
- none

Examples:
User: "Create an IT ticket for VPN access issue"
Output:
{"intent":"create_it_ticket","needs_backend_action":true,"action_name":"create_ticket","arguments":{"category":"IT Support","summary":"VPN access issue"},"reason_for_action":"A ticket must be created in the backend system."}

User: "Find the team and location for ananya"
Output:
{"intent":"lookup_user_record","needs_backend_action":true,"action_name":"lookup_user_record","arguments":{"user_name":"ananya"},"reason_for_action":"The answer should come from the CSV file loaded from S3."}
'''


In [27]:
# Function to extract JSON object from LLM response
def extract_json_object(text: str) -> dict:
    return json.loads(text.strip())

# Function to plan action based on user request
def plan_action(user_request: str) -> dict:
    raw = ask_llm(
        user_prompt=f"{ACTION_PLANNER_PROMPT}\n\nUser request: {user_request}",
        system_prompt="You convert user requests into safe structured action plans.",
        max_tokens=300,
        temperature=0,
        asked_by = "user@123"
    )
    return extract_json_object(raw)

In [28]:
# Test the action planner with a sample user request
plan_action("Create an IT ticket for laptop replacement")

{'intent': 'create_it_ticket',
 'needs_backend_action': True,
 'action_name': 'create_ticket',
 'arguments': {'category': 'IT Support', 'summary': 'laptop replacement'},
 'reason_for_action': 'A ticket must be created in the backend system for laptop replacement.'}

In [29]:
# Test the action planner with another sample user request
plan_action("Find the team and location for ananya")

{'intent': 'lookup_user_record',
 'needs_backend_action': True,
 'action_name': 'lookup_user_record',
 'arguments': {'user_name': 'ananya'},
 'reason_for_action': 'The answer should come from the CSV file loaded from S3.'}

In [30]:
import pandas as pd
# Read data from S3 to be used for backend action execution
S3_DATA_BUCKET = "rag-demo-docs-sri"
S3_FILE_PATH = "training_user_directory.csv"

def read_csv_from_s3(bucket: str, file_path: str) -> pd.DataFrame:
    obj = s3.get_object(Bucket=bucket, Key=file_path)
    return pd.read_csv(obj["Body"])

df = read_csv_from_s3(S3_DATA_BUCKET, S3_FILE_PATH)
print(df.head())

  user_name    full_name                  team   location      manager  \
0    ananya   Ananya Rao      Data Engineering  Bengaluru  Rahul Mehta   
1    vikram  Vikram Shah  Platform Engineering  Hyderabad   Priya Nair   
2      neha   Neha Gupta         QA Automation       Pune  Sandeep Rao   
3     arjun   Arjun Iyer            IT Support    Chennai  Meera Joshi   
4      sara    Sara Khan               Product  Hyderabad    Dev Malik   

     laptop_type  
0    MacBook Air  
1   ThinkPad T14  
2    MacBook Pro  
3    ThinkPad X1  
4  Dell Latitude  


In [31]:
# Function to execute the planned action
def execute_plan(plan: dict) -> dict:
    action_name = plan["action_name"]

    if action_name == "lookup_user_record":
        user_name = plan["arguments"]["user_name"]
        df = read_csv_from_s3(S3_DATA_BUCKET, S3_FILE_PATH)

        match = df[df["user_name"].str.lower() == user_name.lower()]

        if match.empty:
            return {
                "status": "not_found",
                "message": f"No user found for '{user_name}'."
            }

        row = match.iloc[0]

        return {
            "status": "success",
            "message": "User found.",
            "user_record": {
                "user_name": row["user_name"],
                "full_name": row["full_name"],
                "team": row["team"],
                "location": row["location"],
                "manager": row["manager"],
                "laptop_type": row["laptop_type"]
            }
        }

    return {
        "status": "no_action",
        "message": "No backend action required."
    }


In [32]:
# run the full pipeline with a sample user request
def run_prompt_to_action_pipeline(user_request: str) -> dict:
    plan = plan_action(user_request)
    result = execute_plan(plan)
    return {
        "user_request": user_request,
        "plan": plan,
        "result": result
    }

In [33]:
# Test the full pipeline with a sample user request
response = run_prompt_to_action_pipeline("Find the team and location for ananya")
print(json.dumps(response, indent=2))

{
  "user_request": "Find the team and location for ananya",
  "plan": {
    "intent": "lookup_user_record",
    "needs_backend_action": true,
    "action_name": "lookup_user_record",
    "arguments": {
      "user_name": "ananya"
    },
    "reason_for_action": "The answer should come from the CSV file loaded from S3."
  },
  "result": {
    "status": "success",
    "message": "User found.",
    "user_record": {
      "user_name": "ananya",
      "full_name": "Ananya Rao",
      "team": "Data Engineering",
      "location": "Bengaluru",
      "manager": "Rahul Mehta",
      "laptop_type": "MacBook Air"
    }
  }
}


In [34]:
# Test the full pipeline with a sample user request
response = run_prompt_to_action_pipeline("Find the team and location for karthik")
print(json.dumps(response, indent=2))

{
  "user_request": "Find the team and location for karthik",
  "plan": {
    "intent": "lookup_user_record",
    "needs_backend_action": true,
    "action_name": "lookup_user_record",
    "arguments": {
      "user_name": "karthik"
    },
    "reason_for_action": "The answer should come from the CSV file loaded from S3."
  },
  "result": {
    "status": "not_found",
    "message": "No user found for 'karthik'."
  }
}
